# Coverage, budgets, and deadlines

`report.coverage` records what each detector examined, at what scale,
and what was skipped or capped and why. Read it alongside findings to
understand which properties were checked. This chapter explains the coverage
vocabulary, the three budget presets, the optional wall-clock deadline, the
overrides that scale the sampled work, and how detector selection appears in
coverage.

## The coverage vocabulary

`report.coverage` maps each pass to a status and a human-readable detail
line. The built-in passes use these five normal statuses:


In [ ]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Status': ['`ran_exact`', '`ran_sampled`', '`capped`', '`skipped`', '`not_applicable`'], 'Meaning': ['The pass records completed work; check evidence for each finding’s row scope', 'The pass ran on a deterministic sample by design', 'The pass ran but a search cap limited its scope', 'The pass did not run at all', 'No eligible input or applicable result; read the reason'], 'Typical detail': ['"6 cards (exact)"', 'sample size vs table size', 'how many of how many candidates', '"quick tier", "deadline reached", "detector not enabled"', '"0 single-column unique, 0 composite"']})
(GT(_doc_table)
 .fmt_markdown(columns=['Status', 'Meaning', 'Typical detail'])
 .tab_header(title='The coverage vocabulary')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


A `capped` pass performed some work but reached a search limit. For
example, the numeric-pair pass screens all pairs and computes detailed
statistics for the strongest 496. A `skipped` pass did not run. Check the
detail line to see what work supports either status.

## The three budgets

The `budget` argument selects one of three presets. Full-column summaries and the same exact comparisons do not change their
input scope across budgets. Search breadth and which passes run can change.


In [ ]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Setting': ['Column cards, duplicates, keys, sortedness, dates', 'Single-column uniqueness', 'Composite key search', 'Functional dependencies', 'Model passes (screening, tables, rules, regimes)', 'Numeric pair statistics'], '`quick`': ['all exact, full rows', 'exact', '—', '—', '—', '—'], '`standard`': ['same', 'exact', 'pairs, verified up to 16', 'LHS size ≤ 2, verified over ≤ 200,000 rows', '≤ 20,000 encoded rows', '≤ 496 pairs over ≤ 20,000 rows'], '`deep`': ['same', 'exact', 'pairs and triples, verified up to 24', 'LHS size ≤ 3, verified over ≤ 500,000 rows', '≤ 50,000 encoded rows', '≤ 2,016 pairs over ≤ 50,000 rows']})
(GT(_doc_table)
 .fmt_markdown(columns=['Setting', '`quick`', '`standard`', '`deep`'])
 .tab_header(title='The three budgets')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


`quick` runs column summaries and basic integrity checks. `standard`
adds relationship discovery and is the default. `deep` uses wider
searches and larger samples. The limits and sampling methods are in
[](../methodology/sampling.ipynb).


In [ ]:
import random

import polars as pl

import onceover

rng = random.Random(3)
n = 15_000

region_id = [rng.randrange(4) for _ in range(n)]
region_name = [["north", "south", "east", "west"][r] for r in region_id]
quantity = [rng.randrange(1, 21) for _ in range(n)]
double_quantity = [2 * q for q in quantity]
x = [rng.uniform(-1, 1) for _ in range(n)]
y = [v * v for v in x]

df = pl.DataFrame({
    "order_id": list(range(n)),
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
    "x": x,
    "y": y,
})

reports = {b: onceover.profile(df, budget=b) for b in ["quick", "standard", "deep"]}
for b, r in reports.items():
    measured = sum(1 for f in r.findings
                   if any(e["basis"] == "measured" for e in f["evidence"]))
    print(f"{b:9s} {len(r.findings):3d} findings ({measured} with measured evidence)")

`quick` returns full-data summaries and basic integrity findings.
`standard` adds relationships; `deep` uses a wider search but finds the
same structure in this fixture. The coverage table shows which passes
ran at each budget:

In [ ]:
from great_tables import GT, loc, style

passes = sorted(set().union(*[set(r.coverage) for r in reports.values()]))
rows = []
for p in passes:
    row = {"pass": p}
    for b, r in reports.items():
        row[b] = r.coverage.get(p, {"status": "—", "detail": ""})["status"]
    rows.append(row)

(GT(pl.DataFrame(rows))
 .tab_header(title="Pass status by budget", subtitle="15,000 rows × 7 columns")
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"),
            loc.body()))

## Overrides

Three keyword arguments scale the sampled work without touching the exact
passes. `sample_rows` caps the row sample shared by the FD verification, the
encoded matrix, and model fitting. `max_predictors` caps how many predictors
survive screening per fitted model. `max_pairs` caps how many numeric pairs
get detailed statistics after screening. Column cards, duplicate detection,
single-column keys, sortedness, dates, and identity residual verification
always examine the full input.

In [ ]:
default = onceover.profile(df)
tuned = onceover.profile(df, sample_rows=6_000, max_predictors=8, max_pairs=20)

for pass_name in ["encoding", "functional_dependencies", "dependence_screen"]:
    print(f"{pass_name}:")
    print("  default:", default.coverage[pass_name]["detail"])
    print("  tuned:  ", tuned.coverage[pass_name]["detail"])

The tuned run's coverage shows 6,000-row samples where the default used
15,000 (and would use 20,000 on a larger table). Smaller samples are
cheaper but can leave too few rows for model fitting. Check coverage for
passes skipped because the sample was too small. Relationships involving
rare values may also be missed when those values are absent from the sample.

## The deadline

An optional wall-clock budget, `deadline_seconds`, trades completeness for
time. It is checked *between* passes: whichever pass is running when the
deadline passes still completes, and later passes are skipped and recorded
as `"deadline reached"` in coverage. Nothing is interrupted mid-computation,
so any reported finding comes from a pass that completed.

Without a deadline and with `timing=False`, the same package version, data,
and configuration produce byte-identical reports. A deadline makes the
completed work depend on runtime. `timing=True` also makes coverage vary
between runs, by recording elapsed time; it does not change findings.

In [ ]:
import numpy as np

rng_np = np.random.default_rng(5)
big = pl.DataFrame(
    {"id": np.arange(400_000, dtype=np.int64)}
    | {f"x{i:02}": rng_np.random(400_000) for i in range(15)}
)

timed = onceover.profile(big, deadline_seconds=1.0)
deadline_skips = {name: info for name, info in timed.coverage.items()
                  if info["status"] == "skipped"
                  and "deadline" in info["detail"].lower()}
print("Passes skipped because of the deadline:", len(deadline_skips))
for name, info in deadline_skips.items():
    print(f"{name:28s} {info['status']:14s} {info['detail']}")

On this 400,000-row frame, a one-second deadline may skip later passes
if the preceding work takes long enough. A fast run can finish all enabled
passes before the deadline. A `views` entry saying `detector not enabled`
reflects detector selection rather than the deadline. Include coverage
when sharing a report produced under a deadline; it records which passes
completed.

## Detector selection

Selecting a subset with `detectors=[...]` marks deselected passes
`"detector not enabled"`. A detector required by a selected detector still
runs, but its coverage line includes "required as a resource; findings not
selected". Every rule candidate from PRISM or association mining is then
re-counted on a common diagnostic partition; coverage records this check
under `rule_evaluation`. See [](findings.ipynb) for detector selection
examples.

## Reading coverage in practice

- If a finding is absent, check its detector's status. `skipped` means
  the pass did not run; `capped` means it examined only part of its
  candidate search. `ran_exact` is a pass-level label. The detail and each evidence record
  establish which rows and candidates support a particular result.
- For sampled work, read the sample and table sizes in the detail line.
  Each evidence record also gives `scope.population_rows` and
  `scope.rows_evaluated` for its own measurement.
- Before automating an action, check the finding's `basis`, sampling,
  validation, and pass status. Full-data measurements describe the
  current table. Validate sampled or model-based results on the data
  where the action will apply.

## Timings and evidence scope

Use `onceover.profile(df, timing=True)` to add `elapsed_ms` to coverage.
The value measures the interval since the previous coverage entry was
recorded. Setup work between passes can be included; the values are useful
for locating expensive passes rather than measuring allocated memory or
isolating every internal operation.

Coverage keys name pipeline passes, which can differ from detector IDs:
`gbm` appears as `predictability_screen`, for example. Pass status is not a
validation grade. Some passes summarize completion while their findings
use sampled rows; always inspect `evidence.scope`, the evaluated count,
and the detail line. The JSON contract also permits an `error` status.
The [sampling reference](../methodology/sampling.ipynb) lists the methods’
row scopes, including those with independent caps.
